# EDA

탐색적 데이터 분석(Exploratory Data Analysis): 데이터의 분포, 통계량, 관계를 시각화하여 데이터의 특성을 이해하는 과정

- 어떤 머신러닝 모델을 만들기 위해 데이터의 특징(Feature)간의 상관관계를 살펴보기 위해 정리하는 것

In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

df, y = load_wine(as_frame=True, return_X_y=True)
df['quality'] = y
df.info()
print(df.describe())



## Pandas 연습

In [ ]:
# 1. 데이터셋에 포함된 와인의 총 샘플 수 구하기
sample_count = df.shape[0]
sample_count

In [ ]:
# 2. 데이터셋의 특성 수 구하기
feature_count = df.shape[1]
feature_count

In [ ]:
# 3. quality는 어떤 범주가 있는지 구하기
class_count = df['quality'].nunique()
class_count

In [ ]:
# 4. quality 별 샘플 수 구하기 (0, 1, 2가 각각 몇개인지)
class_distribution = df['quality'].value_counts().sort_index()
class_distribution

In [ ]:
# 5. Alcohol의 평균값이 가장 높은 클래스를 구하기
top_alcohol_class = df.groupby('quality')['alcohol'].mean().idxmax()
top_alcohol_class

In [ ]:
# 8. color intensity가 10 이상인 샘플의 비율 - 샘플 확인
filtered_df = df['color_intensity'] >= 10
high_color_samples = df[filtered_df]
high_color_samples

In [ ]:
# 8. color intensity가 10 이상인 샘플의 비율 - 샘플 비율
high_color_ratio = len(high_color_samples) / len(df) * 100
high_color_ratio

In [ ]:
# 10. Proline의 분포에서 가장 높은 피크르 가지는 클래스 번호 구하기
bins = 20  # 몇개 단위로 데이터를 끊을지
global_min, global_max = df['proline'].min(), df['proline'].max()
peak_by_class = {}
for cls in df['quality'].unique():
    counts, _ = np.histogram(df.loc[df['quality'] == cls, 'proline'], bins=bins, range=(global_min, global_max))
    peak_by_class[int(cls)] = counts.max()

print(peak_by_class)


## 상관관계

두 변수가 얼마나 선형적인 관계를 가지는지 나타내는 값

- 항상 -1 과 1 사이
- 1에 가까울 때: 양의 상관관계
- -1에 가까울 때: 음의 상관관계
- 0에 가까울 때: 상관관계가 거의 없음


$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$, $\bar{x}$ (x바): $x$의 평균

예시)
- 기온이 높을 수록 아이스크림이 많이 팔린다 -> 양의 상관관계
- 운동을 많이 할 수록 몸무게가 줄어든다 -> 음의 상관관계


In [ ]:
# 12. Alcohol과 상관관계가 가장 높은 특성 구하기
alcohol_correlations = df.corr()['alcohol']
alcohol_correlations = alcohol_correlations.abs().sort_values(ascending=False)
print(alcohol_correlations)
top_corr_with_alcohol = alcohol_correlations.index[1]
print(top_corr_with_alcohol)
pro_al_corr = df['proline'].corr(df['alcohol'])
print(pro_al_corr)


In [ ]:
# 전체 상관관계 시각화
# 모든 상관관계 저장
corr = df.corr()

fig, ax = plt.subplots(figsize=(11, 8))
n = len(corr)
mask = np.triu(np.ones((n, n)))

sns.heatmap(data=corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask)
ax.grid(False)
ax.set_title('Correlation between features')
ax.set_facecolor("white")
plt.show()


In [ ]:
# Flavanoid 분포
plt.figure(figsize=(6, 4))
sns.histplot(data=df, x='flavanoids', hue='quality', bins=20, kde=True)
plt.title('Flavanoids Distribution')
plt.show()

# 산점도 그리기
sns.scatterplot(data=df, x='flavanoids', y='total_phenols', hue='quality')
plt.show()

In [ ]:
# 높은 상관관계의 특성 5개를 pairplot으로 시각화

# 1. 상관관계가 높은 특성 5개
corr_with_quality = corr['quality'].abs().sort_values(ascending=False)
top_features = corr_with_quality.index[1:6]  # quality 제외, 상위 5개
print(top_features)

plot_df = df[top_features.tolist() + ["quality"]]
sns.pairplot(data=plot_df, hue="quality", corner=True)
plt.show() 

## 결측치 & 이상치 탐지

누락되거나 잘못 기입된 데이터를 탐지하고 처리하는 것

- 결측치: 빠진 데이터, 말그대로 누락된 데이터
- 이상치: 비정상적 데이터, 범위에서 지나치게 크게 벗어난 데이터

In [ ]:
# 난수 고정 (실습이 같은 결과가 나올 수 있도록)
np.random.seed(42)

### 결측치 탐지 및 처리

In [ ]:

# 결측치 생성
df_missing = df.copy()
missing_idx = np.random.choice(df_missing.index, size=10, replace=False)
df_missing.loc[missing_idx, 'flavanoids'] = np.nan

# heatmap으로 결측치 확인
sns.heatmap(df_missing.isnull(), cbar=False)
plt.title('visualize missing values')
plt.show()


In [ ]:

# 결측치는 평균값으로 대체
df_filled = df_missing.fillna(df_missing.mean(numeric_only=True))
print(df_filled.isnull().sum())


### 이상치 탐지 및 처리

In [ ]:

# 0. 이상치 생성
outlier_idx = np.random.choice(df_missing.index, size=5, replace=False)
df_missing.loc[outlier_idx, 'alcohol'] = df_missing['alcohol'].mean() * 5

# 사분위 범위(IQR)로 이상치 탐지
# 전체 데이터를 구간별로 4등분 해서
# 가운데 두 구간의 크기를 IQR로 정의할 때
# 첫번째 구간의 최댓값 - IQR * 1.5
# 세번째 구간의 최솟값 + IQR * 1.5
# 를 벗어나는 데이터를 이상치로 판단
def detect_outliers_iqr(data, column):
    # data = df
    # column = 'alcohol'
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]

outliers_alcohol = detect_outliers_iqr(df_missing, 'alcohol')
outliers_alcohol



In [ ]:
# boxplot으로 이상치 확인
sns.boxplot(x=df_missing['alcohol'])
plt.title('alcohol outliers')
plt.show()


In [ ]:
# 이상치 처리
# 1. 이상치를 제거
df_no_outliers = df_filled[~df_filled.index.isin(outliers_alcohol.index)]
outliers_alcohol = detect_outliers_iqr(df_no_outliers, 'alcohol')
outliers_alcohol

In [ ]:
# 2. 평균값으로 대체
mean_alcohol = df_filled['alcohol'].mean()
df_filled.loc[outliers_alcohol.index, 'alcohol'] = mean_alcohol
outliers_alcohol = detect_outliers_iqr(df_filled, 'alcohol')
outliers_alcohol